In [5]:
!pip install insightface onnxruntime opencv-python torch torchvision matplotlib scikit-learn

  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached ml_dtypes-0.5.4-cp313-cp313-win_amd64.whl.metadata (9.2 kB)
   ---------------------------------------- 0.0/762.2 kB ? eta -:--:--
   ---------------------------------------- 762.2/762.2 kB 6.6 MB/s  0:00:00
   ---------------------------------------- 0.0/13.0 MB ? eta -:--:--
   ---------------------- ----------------- 7.3/13.0 MB 35.8 MB/s eta 0:00:01
   ---------------------------------------- 13.0/13.0 MB 35.1 MB/s  0:00:00
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ------------------------------------- -- 7.6/8.2 MB 37.5 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.2 MB 37.1 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 15.4 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 k


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import insightface
from insightface.model_zoo import get_model
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

Usando dispositivo: cpu


In [8]:
class DepthwiseSeparable(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, 3, stride, 1, groups=in_ch, bias=False)
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.PReLU(out_ch)

    def forward(self, x):
        x = self.relu(self.bn1(self.dw(x)))
        return self.relu(self.bn2(self.pw(x)))

class Bottleneck(nn.Module):
    def __init__(self, in_ch, out_ch, stride, expand):
        super().__init__()
        mid = in_ch * expand
        self.use_res = (stride == 1 and in_ch == out_ch)
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.PReLU(mid),
            nn.Conv2d(mid, mid, 3, stride, 1, groups=mid, bias=False),
            nn.BatchNorm2d(mid),
            nn.PReLU(mid),
            nn.Conv2d(mid, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch)
        )

    def forward(self, x):
        out = self.block(x)
        return out + x if self.use_res else out

In [9]:
class MobileFaceNetBackbone(nn.Module):
    def __init__(self, embedding_size=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.PReLU(64),
            DepthwiseSeparable(64, 64),
            Bottleneck(64, 64,  stride=2, expand=2),
            Bottleneck(64, 64,  stride=1, expand=2),
            Bottleneck(64, 64,  stride=1, expand=2),
            Bottleneck(64, 64,  stride=1, expand=2),
            Bottleneck(64, 128, stride=2, expand=4),
            Bottleneck(128, 128, stride=1, expand=2),
            Bottleneck(128, 128, stride=1, expand=2),
            Bottleneck(128, 128, stride=1, expand=2),
            Bottleneck(128, 128, stride=1, expand=2),
            Bottleneck(128, 128, stride=1, expand=2),
            Bottleneck(128, 128, stride=1, expand=2),
            Bottleneck(128, 256, stride=2, expand=4),
            Bottleneck(256, 256, stride=1, expand=2),
            Bottleneck(256, 256, stride=1, expand=2),
            nn.Conv2d(256, 512, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.PReLU(512),
        )
        # Global Depthwise Conv (GDConv)
        self.gdconv = nn.Conv2d(512, 512, 7, groups=512, bias=False)
        self.bn_gd  = nn.BatchNorm2d(512)
        self.fc     = nn.Linear(512, embedding_size)
        self.bn_fc  = nn.BatchNorm1d(embedding_size)

    def forward(self, x):
        x = self.features(x)
        x = self.bn_gd(self.gdconv(x))
        x = x.view(x.size(0), -1)
        x = self.bn_fc(self.fc(x))
        return x

backbone = MobileFaceNetBackbone(embedding_size=128).to(device)
print("Backbone MobileFaceNet creado")
print(f"Parámetros totales: {sum(p.numel() for p in backbone.parameters()):,}")

Backbone MobileFaceNet creado
Parámetros totales: 1,521,792


In [10]:
class MobileFaceNetClassifier(nn.Module):
    def __init__(self, backbone, num_classes=40, freeze_backbone=True):
        super().__init__()
        self.backbone = backbone

        # Congelar backbone en fase 1
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.PReLU(64),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        embedding = self.backbone(x)
        return self.classifier(embedding)

    def unfreeze_last_n(self, n=3):
        """Descongela las últimas n capas del backbone para fine-tuning"""
        layers = list(self.backbone.features.children())
        for layer in layers[-n:]:
            for param in layer.parameters():
                param.requires_grad = True
        print(f"Últimas {n} capas del backbone descongeladas")

    def unfreeze_all(self):
        for param in self.backbone.parameters():
            param.requires_grad = True
        print("Backbone completamente descongelado")

model = MobileFaceNetClassifier(backbone, num_classes=40, freeze_backbone=True).to(device)
print(model)

MobileFaceNetClassifier(
  (backbone): MobileFaceNetBackbone(
    (features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): PReLU(num_parameters=64)
      (3): DepthwiseSeparable(
        (dw): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (pw): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): PReLU(num_parameters=64)
      )
      (4): Bottleneck(
        (block): Sequential(
          (0): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, tr

In [23]:
class ORLDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        EXTENSIONES = ('.pgm', '.png', '.jpg', '.jpeg', '.bmp', '.gif')

        sujetos = sorted([s for s in os.listdir(root_dir) 
                          if os.path.isdir(os.path.join(root_dir, s))])
        
        for label, subject in enumerate(sujetos):
            subject_path = os.path.join(root_dir, subject)
            images = sorted([f for f in os.listdir(subject_path)
                             if f.lower().endswith(EXTENSIONES)])
            for img_name in images:
                self.samples.append((os.path.join(subject_path, img_name), label))

        print(f"  → {root_dir}: {len(sujetos)} sujetos, {len(self.samples)} imágenes cargadas")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# Transforms para cada combinación
transform_base = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

transform_aug = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])


PATHS = {
    'LR_x2':    ('dataset/Training_LR_escala2/',                  'dataset/Testing/'),
    'CLAHE_x2': ('dataset/CLAHE/Training_CLAHE_escala2/',         'dataset/Testing/'),
    'SRCNN_x2': ('dataset/Super_resolution/Training_SR_escala2/', 'dataset/Testing/'),
    'LR_x4':    ('dataset/Training_LR_escala4/',                  'dataset/Testing/'),
    'CLAHE_x4': ('dataset/CLAHE/Training_CLAHE_escala4/',         'dataset/Testing/'),
    'SRCNN_x4': ('dataset/Super_resolution/Training_SR_escala4/', 'dataset/Testing/'),
}

In [24]:
def train_model(model, train_loader, val_loader, epochs=50, lr=1e-3, phase_name="Fase1", verbose=False):
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_acc = 0.0

    for epoch in range(epochs):
        # --- Entrenamiento ---
        model.train()
        train_loss, train_preds, train_labels = 0, [], []

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_preds.extend(outputs.argmax(1).cpu().numpy())
            train_labels.extend(labels.cpu().numpy())

        # --- Validación ---
        model.eval()
        val_loss, val_preds, val_labels = 0, [], []

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                val_preds.extend(outputs.argmax(1).cpu().numpy())
                val_labels.extend(labels.cpu().numpy())

        # Métricas
        t_loss = train_loss / len(train_loader)
        v_loss = val_loss  / len(val_loader)
        t_acc  = accuracy_score(train_labels, train_preds)
        v_acc  = accuracy_score(val_labels,   val_preds)

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)

        scheduler.step(v_loss)

        # Obtener lr actual
        current_lr = optimizer.param_groups[0]['lr']

        if best_acc < v_acc:
            best_acc = v_acc
            torch.save(model.state_dict(), f'mobilefacenet_{phase_name}_best.pth')
            saved = " ✓ guardado"
        else:
            saved = ""

        # Verbose por época
        if verbose:
            print(f"{epoch+1:<8} {t_loss:<12.4f} {v_loss:<12.4f} {t_acc:<12.4f} {v_acc:<10.4f} {current_lr:.2e}{saved}")

    print(f"\nMejor accuracy validación [{phase_name}]: {best_acc:.4f}")
    return history

In [26]:
resultados = {}

for nombre, (ruta_train, ruta_test) in PATHS.items():
    print(f"\n{'='*50}")
    print(f"Entrenando con: {nombre.upper()}")
    print(f"  Train: {ruta_train}")
    print(f"  Test:  {ruta_test}")
    print(f"{'='*50}")

    train_ds = ORLDataset(ruta_train, transform=transform_aug)
    val_ds   = ORLDataset(ruta_test,  transform=transform_base)

    print(f"Imágenes train: {len(train_ds)} | Imágenes val: {len(val_ds)}")

    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=0, drop_last=False)

    backbone_i = MobileFaceNetBackbone(embedding_size=128).to(device)
    model_i    = MobileFaceNetClassifier(backbone_i, num_classes=40, freeze_backbone=True).to(device)

    # Solo transfer learning — backbone congelado todo el tiempo
    print("\n--- TRANSFER LEARNING: Backbone congelado ---")
    print(f"{'Época':<8} {'Train Loss':<12} {'Val Loss':<12} {'Train Acc':<12} {'Val Acc':<10} {'LR'}")
    print("-" * 65)
    history = train_model(model_i, train_loader, val_loader,
                          epochs=50, lr=1e-3,
                          phase_name=f"{nombre}",
                          verbose=True)

    resultados[nombre] = history


Entrenando con: LR_X2
  Train: dataset/Training_LR_escala2/
  Test:  dataset/Testing/
  → dataset/Training_LR_escala2/: 40 sujetos, 360 imágenes cargadas
  → dataset/Testing/: 40 sujetos, 40 imágenes cargadas
Imágenes train: 360 | Imágenes val: 40

--- TRANSFER LEARNING: Backbone congelado ---
Época    Train Loss   Val Loss     Train Acc    Val Acc    LR
-----------------------------------------------------------------
1        3.7298       3.6481       0.0222       0.0250     1.00e-03 ✓ guardado
2        3.6405       3.6418       0.0528       0.0000     1.00e-03
3        3.5466       3.6082       0.0861       0.0250     1.00e-03
4        3.4826       3.5554       0.1056       0.0750     1.00e-03 ✓ guardado
5        3.4306       3.5456       0.1444       0.0500     1.00e-03
6        3.3191       3.5011       0.1417       0.0500     1.00e-03
7        3.2649       3.4666       0.1722       0.1500     1.00e-03 ✓ guardado
8        3.2035       3.4392       0.1639       0.1750     1.00e-03